# 2. Service-to-Service with Client Credentials

**Scenario**: a background daemon (cron job, worker, scheduled function) needs to call an API. There's no user sitting in front of a browser. This is the canonical *service-to-service* (S2S) pattern.

The flow is called **client credentials**:

```
daemon ──(client_id + client_secret)──▶ Entra ID  ──▶ access token
daemon ──(Bearer token)─────────────────▶ api-b    ──▶ data
```

Entra treats the daemon like any other principal. It checks:
1. Does the secret match?
2. Has the daemon's service principal been **granted** the app-role it's asking for, on the target API?

The token it issues carries the granted roles in the `roles` claim.

## The `/.default` scope

When you do client credentials, you can't pick individual scopes — you ask for `<resource>/.default`, meaning *"give me everything this app has been granted on this resource"*. That's an Entra-ism: delegated scopes need user consent, so for the app-only case the `/.default` trick hands back all pre-consented app-roles.

In [ ]:
import httpx, json, base64

TOKEN_URL = 'http://localhost:9100/contoso/oauth2/v2.0/token'
API_B     = 'http://localhost:8002'

# ⛔ INSPECTION ONLY: no signature check, no claim validation. See notebook 1.
# We use this to *look at* tokens; api-b uses common/auth.py to *trust* them.
def decode_payload(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- 1. Get a token as the daemon app ---
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
token = r.json()['access_token']
claims = decode_payload(token)
print(json.dumps(claims, indent=2))

# The three properties the prose below claims. Fail loudly if any stops holding.
assert claims['aud'] == 'api://api-b', f"token must be scoped to api-b, got aud={claims['aud']}"
assert claims['roles'] == ['Files.Read.All'], f"expected the granted app role, got {claims.get('roles')}"
assert 'upn' not in claims and 'scp' not in claims, (
    f"client_credentials is app-only: there must be no user claims, got "
    f"upn={claims.get('upn')!r} scp={claims.get('scp')!r}"
)

Notice:
- `aud = api://api-b` — token is *for* API-B, not usable elsewhere.
- `roles = ['Files.Read.All']` — the app role granted to the daemon.
- **No `upn` / `scp`** — this is an app-only token; no user was involved.

## 2. Call API-B

In [ ]:
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {token}'})
print(r.status_code)
print(json.dumps(r.json(), indent=2))

assert r.status_code == 200, f'daemon holds Files.Read.All, expected 200, got {r.status_code}'
body = r.json()
assert body['mode'] == 'app-only', f"expected app-only mode, got {body['mode']}"
# App-only means "no user to scope to", so api-b returns every file, including bob's.
assert len(body['files']) == 3, f"app-only should see all 3 files, got {len(body['files'])}"

## 3. What happens if the app role was **not granted**?

A `client_credentials` token succeeds as long as the secret is valid — but the token only carries the roles the **service principal was granted** on the target resource. If the role is missing, you still get a token, but the target API will reject the call with **403**.

We seeded a second daemon, `reporting-daemon`, that has **no** granted roles on `api-b`. Let's see what happens.


In [ ]:
# Token issues fine (secret is valid), but the `roles` claim will be empty.
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'reporting-daemon-client-id',
    'client_secret': 'reporting-daemon-secret-value',
    'scope': 'api://api-b/.default',
})
assert r.status_code == 200, f'authentication should succeed - only authorization is missing (got {r.status_code})'
no_role_token = r.json()['access_token']
print('roles claim:', decode_payload(no_role_token).get('roles'))
assert not decode_payload(no_role_token).get('roles'), 'reporting-daemon is seeded with no granted roles'

# API-B rejects: no Files.Read.All role.
r2 = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {no_role_token}'})
print('status:', r2.status_code, r2.json())
assert r2.status_code == 403, (
    f'authenticated-but-unauthorized must be 403, not 401 and never 200 - got {r2.status_code}'
)

# Bonus: what about a wrong secret? Entra refuses at the token endpoint itself.
r3 = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'WRONG',
    'scope': 'api://api-b/.default',
})
print('bad secret ->', r3.status_code, r3.json())
assert r3.status_code == 401, f'a bad secret must fail at the IdP, got {r3.status_code}'
assert 'access_token' not in r3.text, 'the IdP must not hand out a token to a caller it could not authenticate'

## 4. What happens with a token for the wrong API?

Tokens include `aud` (audience). API-B rejects anything that isn't `api://api-b`.

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-a/.default',   # wrong target
})
other = r.json().get('access_token')
assert decode_payload(other)['aud'] == 'api://api-a', 'this token is deliberately minted for the wrong API'

r2 = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {other}'})
print('status:', r2.status_code, r2.json())
assert r2.status_code == 401, (
    f'a token whose aud is another API must be rejected at authentication time (401), got {r2.status_code}. '
    'If this ever returns 200, audience validation is off and every API in the tenant is a replay target.'
)

That's the whole point of `aud` — a stolen token from one API can't be replayed against another.

## 5. Using MSAL (what you'd actually write in production)

Nobody POSTs to the token endpoint by hand in real code — you use Microsoft's **MSAL** library. It handles caching, refresh, retries, certificate auth, etc.

**MSAL snippet (for real Entra, not runnable against our mock — MSAL rejects non-HTTPS authorities):**

```python
from msal import ConfidentialClientApplication

app = ConfidentialClientApplication(
    client_id=os.environ['AZURE_CLIENT_ID'],
    client_credential=os.environ['AZURE_CLIENT_SECRET'],
    authority=f"https://login.microsoftonline.com/{os.environ['AZURE_TENANT_ID']}",
)
result = app.acquire_token_for_client(scopes=['api://api-b/.default'])
token = result['access_token']
```

MSAL adds caching, automatic refresh, certificate auth and confidential-client flows on top
of the raw HTTP calls we used above.

## Secret vs certificate — pick the certificate

`client_credential` above takes either. They are *not* equivalent:

| | Client secret | Certificate |
|-|---------------|-------------|
| What travels to Entra | The secret itself, on every token request | A short-lived JWT **signed** by your private key (`client_assertion`) — the key never leaves you |
| If the transcript/proxy log leaks | Attacker has the credential | Attacker has a spent, expired assertion |
| Max lifetime in Entra | 24 months (and tenants increasingly cap this at 6) | Up to the cert's validity |
| Storage | A string in Key Vault / env var | Key Vault certificate, or the platform key store |
| Rotation | Manual, and everything breaks at once when you forget | Same problem, but automatable via Key Vault |

```python
app = ConfidentialClientApplication(
    client_id=os.environ['AZURE_CLIENT_ID'],
    client_credential={
        'thumbprint': os.environ['AZURE_CLIENT_CERTIFICATE_THUMBPRINT'],
        'private_key': open(os.environ['AZURE_CLIENT_CERTIFICATE_PATH']).read(),
    },
    authority=f"https://login.microsoftonline.com/{os.environ['AZURE_TENANT_ID']}",
)
```

Order of preference, best first: **managed identity** (no credential at all, notebook 3) →
**workload identity federation** (no stored credential, notebook 5) → **certificate** →
**client secret**. Our mock only implements the secret case because it is the one you can
demonstrate in twenty lines.

## Token caching — do not fetch one per request

`acquire_token_for_client` checks MSAL's in-memory cache first and only calls Entra when the
cached token is within ~5 minutes of `exp`. Keep **one** `ConfidentialClientApplication`
alive for the process lifetime; constructing a new one per request throws the cache away and
gives you a token request per API call — which is both slow and a fast route to Entra
throttling you (`AADSTS900023` / HTTP 429).

Note what client credentials does **not** give you: there is **no refresh token**. There is
nothing to refresh — the app still holds its own credential, so it just asks for a new access
token. Refresh tokens exist for flows where the user is no longer present to re-authenticate.

## Summary

- **Client credentials** = app authenticates with its own creds, gets an app-only token.
- Token carries `roles`, *not* `scp` — these are application permissions.
- Always call `<resource>/.default` for client credentials.
- Audience (`aud`) scoping stops token replay across APIs; a missing role is a **403**, a
  wrong audience or bad signature is a **401**.
- A valid secret gets you a token; it does not get you *access*. Authentication and
  authorization fail at different layers, and the notebook asserts both.
- Use MSAL, not raw HTTP, in real code — and prefer a certificate over a secret, or better,
  **managed identity** (notebook 3) so there is no credential to leak at all.